# Descriptor multitask — протокол как Cell 6

Тот же пайплайн, что `comparison_old_new_multitask.ipynb` (Cell 6):
- **5-fold `KFold` по строкам** multitask-датасета (не GroupKFold по молекуле)
- Baseline regression / classification per target
- Multitask + **org/met** (categorical) и **OpenAI** text embeddings
- Дополнительно: multitask + **OneHot** task_id (как Cell 3 на old data)

Вместо 256d encoder — **15 hand-crafted descriptors** (или 12 на biocides, без dipoles).

Переключатель `DATASET` в следующей ячейке: `biocides` (как Cell 6, 108 mol) или `conc` (82 mol, все 15 descr).

In [ ]:
import pickle
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, CatBoostRegressor, Pool
from scipy.stats import pearsonr
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')

BASE = Path('/Users/egorilin/Desktop/COMA')
RANDOM_STATE = 42
N_FOLDS = 5

# 'biocides' — как Cell 6 (108 mol); 'conc' — все 15 descr (82 mol)
DATASET = 'biocides'

FEATURE_COLS_FULL = [
    'Molecular weight', 'LogP', 'mmff94_dipole', 'gasteiger_dipole', 'eem2015bm_dipole',
    'Molformer_157', 'Molformer_197', 'Molformer_668', 'Molformer_85',
    'Molformer_127', 'Molformer_732', 'Molformer_543', 'Molformer_729',
    'Molformer_337', 'Molformer_686',
]
FEATURE_COLS_BIO = [
    'Molecular weight', 'LogP',
    'Molformer_157', 'Molformer_197', 'Molformer_668', 'Molformer_85',
    'Molformer_127', 'Molformer_732', 'Molformer_543', 'Molformer_729',
    'Molformer_337', 'Molformer_686',
]

TARGET_COLUMNS = [
    'S. aureus ATCC 43300 Activity MIC, мг/л',
    'S. aureus ATCC 43300 Activity MBC, мг/л',
    'E. coli ATCC 25922 Activity MIC, мг/л',
    'E. coli ATCC 25922 Activity MBC, мг/л',
]

CLASS_DESCRIPTIONS = {
    'S. aureus ATCC 43300_MIC': (
        'Minimum Inhibitory Concentration against Staphylococcus aureus ATCC 43300, '
        'a methicillin-resistant Gram-positive bacterium. MIC measures the lowest '
        'concentration that inhibits visible bacterial growth.'
    ),
    'S. aureus ATCC 43300_MBC': (
        'Minimum Bactericidal Concentration against Staphylococcus aureus ATCC 43300, '
        'a methicillin-resistant Gram-positive bacterium. MBC measures the lowest '
        'concentration that kills 99.9% of bacteria.'
    ),
    'E. coli ATCC 25922_MIC': (
        'Minimum Inhibitory Concentration against Escherichia coli ATCC 25922, '
        'a Gram-negative reference strain. MIC measures the lowest concentration '
        'that inhibits visible bacterial growth.'
    ),
    'E. coli ATCC 25922_MBC': (
        'Minimum Bactericidal Concentration against Escherichia coli ATCC 25922, '
        'a Gram-negative reference strain. MBC measures the lowest concentration '
        'that kills 99.9% of bacteria.'
    ),
}

CB_KW = dict(iterations=500, learning_rate=0.05, depth=6, verbose=False, random_seed=RANDOM_STATE)
print(f'Dataset mode: {DATASET}')

In [ ]:
# OpenAI class embeddings — из кэша (как regression_interpretability.py)
cache_path = BASE / 'checkpoints' / 'regression' / 'class_emb_openai.pkl'
if not cache_path.exists():
    raise FileNotFoundError(
        f'Нет {cache_path}. Запустите regression_interpretability.py --mode openai '
        'или comparison_old_new_multitask Cell 6 с API-ключом.'
    )
with open(cache_path, 'rb') as f:
    class_emb_dict = pickle.load(f)
print(f'✓ OpenAI class emb cache: {len(class_emb_dict)} keys, dim={len(next(iter(class_emb_dict.values())))}')

In [ ]:
def load_descriptor_matrix(dataset: str):
    if dataset == 'conc':
        df = pd.read_excel(BASE / 'Data_molformer_conc.xlsx')
        feat = FEATURE_COLS_FULL
        X = df[feat].values.astype(float)
        target_df = df
        note = '15 descriptors (full)'
    elif dataset == 'biocides':
        meta = pd.read_excel(BASE / 'Data_biocides.xlsx')
        mf = pd.read_excel(BASE / 'Test_molformer_original_biocides.xlsx').drop(columns=['Unnamed: 0'])
        if len(meta) != len(mf):
            raise ValueError(f'Размеры biocides: meta={len(meta)} mf={len(mf)}')
        feat = FEATURE_COLS_BIO
        merged = meta[['Molecular weight', 'LogP']].copy()
        for c in feat[2:]:
            merged[c] = mf[c].values
        X = merged[feat].values.astype(float)
        target_df = meta
        note = '12 descriptors (biocides: без dipole-колонок)'
    else:
        raise ValueError(dataset)
    return X, target_df, feat, note

X_mol, target_bio, feature_cols, data_note = load_descriptor_matrix(DATASET)
print(f'✓ {data_note}')
print(f'  molecules: {X_mol.shape[0]}, descriptor dim: {X_mol.shape[1]}')

In [ ]:
# ─── Построение датасетов (как Cell 6, часть 4) ───────────────────────────
print('=' * 70)
print('ПОСТРОЕНИЕ ДАТАСЕТОВ')
print('=' * 70)

per_target = {}
for col in TARGET_COLUMNS:
    valid = ~target_bio[col].isna()
    organism = col.split(' Activity ')[0]
    metric = col.split(' Activity ')[1].replace(', мг/л', '')
    short = f"{organism.split(' ')[0]}_{metric}"

    per_target[short] = {
        'X': X_mol[valid],
        'y_reg': np.log2(target_bio[col][valid].values.astype(float)),
        'y_cls': LabelEncoder().fit_transform(target_bio[col][valid].values),
        'le': LabelEncoder().fit(target_bio[col][valid].values),
        'n': int(valid.sum()),
        'organism': organism,
        'metric': metric,
    }
    print(f"  {short}: {per_target[short]['n']} samples, classes={len(per_target[short]['le'].classes_)}")

mt_rows = []
for col in TARGET_COLUMNS:
    valid = ~target_bio[col].isna()
    X_v = X_mol[valid]
    y_v = np.log2(target_bio[col][valid].values.astype(float))
    y_raw = target_bio[col][valid].values.astype(float)
    organism = col.split(' Activity ')[0]
    metric = col.split(' Activity ')[1].replace(', мг/л', '')
    key = f'{organism}_{metric}'
    task_idx = TARGET_COLUMNS.index(col)

    for i in range(len(X_v)):
        mt_rows.append({
            'mol': X_v[i], 'y_reg': y_v[i], 'y_raw': y_raw[i],
            'organism': organism, 'metric': metric, 'key': key, 'task_idx': task_idx,
        })

print(f'\nMulti-task dataset: {len(mt_rows)} rows')

X_mt_mol = np.array([r['mol'] for r in mt_rows])
y_mt_reg = np.array([r['y_reg'] for r in mt_rows])
y_mt_raw = np.array([r['y_raw'] for r in mt_rows])
task_idx_arr = np.array([r['task_idx'] for r in mt_rows])
n_tasks = len(TARGET_COLUMNS)

org_enc = LabelEncoder(); met_enc = LabelEncoder()
org_coded = org_enc.fit_transform([r['organism'] for r in mt_rows])
met_coded = met_enc.fit_transform([r['metric'] for r in mt_rows])

# OneHot (доп. к Cell 6)
X_mt_onehot = np.hstack([X_mt_mol, np.eye(n_tasks)[task_idx_arr]])

# Categorical org/met
X_mt_cat = pd.DataFrame(X_mt_mol, columns=[f'd{i}' for i in range(X_mt_mol.shape[1])])
X_mt_cat['org'] = [str(x) for x in org_coded]
X_mt_cat['met'] = [str(x) for x in met_coded]

# OpenAI
oai_features = np.array([class_emb_dict[r['key']] for r in mt_rows])
X_mt_oai = np.hstack([X_mt_mol, oai_features])

comp_labels = [f"{r['key']}_{r['y_raw']:.1f}" for r in mt_rows]
comp_enc = LabelEncoder()
y_mt_cls = comp_enc.fit_transform(comp_labels)

print(f'  OneHot features: {X_mt_onehot.shape}')
print(f'  Categorical features: {X_mt_cat.shape}')
print(f'  OpenAI features: {X_mt_oai.shape}')
print(f'  Composite classes: {len(comp_enc.classes_)}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# REGRESSION — 5-fold KFold по строкам (как Cell 6)
# ═══════════════════════════════════════════════════════════════════════════
print('\n' + '=' * 70)
print(f'REGRESSION ({N_FOLDS}-fold KFold on rows, {DATASET})')
print('=' * 70)

# A. Baseline
print('\n' + '─' * 60)
print('A. BASELINE — каждый таргет отдельно')
print('─' * 60)

baseline_reg = {}
for short, data in per_target.items():
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    fold_m = {'mae': [], 'rmse': [], 'r2': [], 'pearson': []}
    for tr_idx, val_idx in kf.split(data['X']):
        cb = CatBoostRegressor(**CB_KW)
        cb.fit(data['X'][tr_idx], data['y_reg'][tr_idx])
        yp = cb.predict(data['X'][val_idx])
        yt = data['y_reg'][val_idx]
        fold_m['mae'].append(mean_absolute_error(yt, yp))
        fold_m['rmse'].append(np.sqrt(mean_squared_error(yt, yp)))
        fold_m['r2'].append(r2_score(yt, yp))
        pr, _ = pearsonr(yt, yp) if len(yt) > 2 else (0, 1)
        fold_m['pearson'].append(pr)
    baseline_reg[short] = {m: (np.mean(v), np.std(v)) for m, v in fold_m.items()}
    print(f"  {short:20s} | R2={baseline_reg[short]['r2'][0]:.4f}±{baseline_reg[short]['r2'][1]:.4f} "
          f"MAE={baseline_reg[short]['mae'][0]:.4f}±{baseline_reg[short]['mae'][1]:.4f}")

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

# B. Multitask + OneHot
print('\n' + '─' * 60)
print('B. MULTI-TASK + OneHot task_id')
print('─' * 60)
mt_onehot_reg = {'mae': [], 'rmse': [], 'r2': [], 'pearson': []}
for tr_idx, val_idx in kf.split(X_mt_onehot):
    cb = CatBoostRegressor(**CB_KW)
    cb.fit(X_mt_onehot[tr_idx], y_mt_reg[tr_idx])
    yp = cb.predict(X_mt_onehot[val_idx])
    yt = y_mt_reg[val_idx]
    mt_onehot_reg['mae'].append(mean_absolute_error(yt, yp))
    mt_onehot_reg['rmse'].append(np.sqrt(mean_squared_error(yt, yp)))
    mt_onehot_reg['r2'].append(r2_score(yt, yp))
    pr, _ = pearsonr(yt, yp)
    mt_onehot_reg['pearson'].append(pr)
mt_onehot_summary = {m: (np.mean(v), np.std(v)) for m, v in mt_onehot_reg.items()}
print(f"  R2={mt_onehot_summary['r2'][0]:.4f}±{mt_onehot_summary['r2'][1]:.4f} "
      f"MAE={mt_onehot_summary['mae'][0]:.4f}±{mt_onehot_summary['mae'][1]:.4f}")

# C. Multitask + org/met (как Cell 6)
print('\n' + '─' * 60)
print('C. MULTI-TASK + org/met categorical')
print('─' * 60)
mt_cat_reg = {'mae': [], 'rmse': [], 'r2': [], 'pearson': []}
for tr_idx, val_idx in kf.split(X_mt_cat):
    pool_tr = Pool(X_mt_cat.iloc[tr_idx], y_mt_reg[tr_idx], cat_features=['org', 'met'])
    cb = CatBoostRegressor(**CB_KW)
    cb.fit(pool_tr)
    yp = cb.predict(X_mt_cat.iloc[val_idx])
    yt = y_mt_reg[val_idx]
    mt_cat_reg['mae'].append(mean_absolute_error(yt, yp))
    mt_cat_reg['rmse'].append(np.sqrt(mean_squared_error(yt, yp)))
    mt_cat_reg['r2'].append(r2_score(yt, yp))
    pr, _ = pearsonr(yt, yp)
    mt_cat_reg['pearson'].append(pr)
mt_cat_reg_summary = {m: (np.mean(v), np.std(v)) for m, v in mt_cat_reg.items()}
print(f"  R2={mt_cat_reg_summary['r2'][0]:.4f}±{mt_cat_reg_summary['r2'][1]:.4f} "
      f"MAE={mt_cat_reg_summary['mae'][0]:.4f}±{mt_cat_reg_summary['mae'][1]:.4f}")

# D. Multitask + OpenAI (как Cell 6)
print('\n' + '─' * 60)
print('D. MULTI-TASK + OpenAI text embeddings')
print('─' * 60)
mt_oai_reg = {'mae': [], 'rmse': [], 'r2': [], 'pearson': []}
for tr_idx, val_idx in kf.split(X_mt_oai):
    cb = CatBoostRegressor(**CB_KW)
    cb.fit(X_mt_oai[tr_idx], y_mt_reg[tr_idx])
    yp = cb.predict(X_mt_oai[val_idx])
    yt = y_mt_reg[val_idx]
    mt_oai_reg['mae'].append(mean_absolute_error(yt, yp))
    mt_oai_reg['rmse'].append(np.sqrt(mean_squared_error(yt, yp)))
    mt_oai_reg['r2'].append(r2_score(yt, yp))
    pr, _ = pearsonr(yt, yp)
    mt_oai_reg['pearson'].append(pr)
mt_oai_reg_summary = {m: (np.mean(v), np.std(v)) for m, v in mt_oai_reg.items()}
print(f"  R2={mt_oai_reg_summary['r2'][0]:.4f}±{mt_oai_reg_summary['r2'][1]:.4f} "
      f"MAE={mt_oai_reg_summary['mae'][0]:.4f}±{mt_oai_reg_summary['mae'][1]:.4f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CLASSIFICATION — как Cell 6
# ═══════════════════════════════════════════════════════════════════════════
print('\n' + '=' * 70)
print(f'CLASSIFICATION ({N_FOLDS}-fold, {DATASET})')
print('=' * 70)

print('\n' + '─' * 60)
print('E. BASELINE CLASSIFICATION — каждый таргет отдельно')
print('─' * 60)

baseline_cls = {}
for short, data in per_target.items():
    min_count = min(Counter(data['y_cls']).values())
    if min_count >= N_FOLDS:
        kf_c = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
        splits = kf_c.split(data['X'], data['y_cls'])
    else:
        kf_c = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
        splits = kf_c.split(data['X'])

    fold_m = {'acc': [], 'f1_macro': [], 'f1_weighted': []}
    for tr_idx, val_idx in splits:
        cb = CatBoostClassifier(**CB_KW, auto_class_weights='Balanced')
        cb.fit(data['X'][tr_idx], data['y_cls'][tr_idx])
        yp = cb.predict(data['X'][val_idx]).flatten().astype(int)
        yt = data['y_cls'][val_idx]
        fold_m['acc'].append(accuracy_score(yt, yp))
        fold_m['f1_macro'].append(f1_score(yt, yp, average='macro', zero_division=0))
        fold_m['f1_weighted'].append(f1_score(yt, yp, average='weighted', zero_division=0))

    baseline_cls[short] = {m: (np.mean(v), np.std(v)) for m, v in fold_m.items()}
    print(f"  {short:20s} | Acc={baseline_cls[short]['acc'][0]:.4f}±{baseline_cls[short]['acc'][1]:.4f} "
          f"F1w={baseline_cls[short]['f1_weighted'][0]:.4f}±{baseline_cls[short]['f1_weighted'][1]:.4f}")

print('\n' + '─' * 60)
print('F. MULTI-TASK CLASSIFICATION + org/met')
print('─' * 60)
mt_cat_cls = {'acc': [], 'f1_macro': [], 'f1_weighted': []}
for tr_idx, val_idx in KFold(N_FOLDS, shuffle=True, random_state=RANDOM_STATE).split(X_mt_cat):
    pool_tr = Pool(X_mt_cat.iloc[tr_idx], y_mt_cls[tr_idx], cat_features=['org', 'met'])
    cb = CatBoostClassifier(**CB_KW, auto_class_weights='Balanced')
    cb.fit(pool_tr)
    yp = cb.predict(X_mt_cat.iloc[val_idx]).flatten().astype(int)
    yt = y_mt_cls[val_idx]
    mt_cat_cls['acc'].append(accuracy_score(yt, yp))
    mt_cat_cls['f1_macro'].append(f1_score(yt, yp, average='macro', zero_division=0))
    mt_cat_cls['f1_weighted'].append(f1_score(yt, yp, average='weighted', zero_division=0))
mt_cat_cls_s = {m: (np.mean(v), np.std(v)) for m, v in mt_cat_cls.items()}
print(f"  Acc={mt_cat_cls_s['acc'][0]:.4f}±{mt_cat_cls_s['acc'][1]:.4f} "
      f"F1w={mt_cat_cls_s['f1_weighted'][0]:.4f}±{mt_cat_cls_s['f1_weighted'][1]:.4f}")

print('\n' + '─' * 60)
print('G. MULTI-TASK CLASSIFICATION + OpenAI')
print('─' * 60)
mt_oai_cls = {'acc': [], 'f1_macro': [], 'f1_weighted': []}
for tr_idx, val_idx in KFold(N_FOLDS, shuffle=True, random_state=RANDOM_STATE).split(X_mt_oai):
    cb = CatBoostClassifier(**CB_KW, auto_class_weights='Balanced')
    cb.fit(X_mt_oai[tr_idx], y_mt_cls[tr_idx])
    yp = cb.predict(X_mt_oai[val_idx]).flatten().astype(int)
    yt = y_mt_cls[val_idx]
    mt_oai_cls['acc'].append(accuracy_score(yt, yp))
    mt_oai_cls['f1_macro'].append(f1_score(yt, yp, average='macro', zero_division=0))
    mt_oai_cls['f1_weighted'].append(f1_score(yt, yp, average='weighted', zero_division=0))
mt_oai_cls_s = {m: (np.mean(v), np.std(v)) for m, v in mt_oai_cls.items()}
print(f"  Acc={mt_oai_cls_s['acc'][0]:.4f}±{mt_oai_cls_s['acc'][1]:.4f} "
      f"F1w={mt_oai_cls_s['f1_weighted'][0]:.4f}±{mt_oai_cls_s['f1_weighted'][1]:.4f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# СВОДНЫЕ ТАБЛИЦЫ + сохранение (как Cell 6)
# ═══════════════════════════════════════════════════════════════════════════
print('\n' + '=' * 70)
print('СВОДНЫЕ ТАБЛИЦЫ')
print('=' * 70)

bl_r2 = np.mean([v['r2'][0] for v in baseline_reg.values()])
bl_mae = np.mean([v['mae'][0] for v in baseline_reg.values()])
bl_rmse = np.mean([v['rmse'][0] for v in baseline_reg.values()])
bl_pr = np.mean([v['pearson'][0] for v in baseline_reg.values()])

print('\n📈 REGRESSION (mean ± std across folds):')
print(f"{'Approach':45s} {'R2':>16s} {'MAE':>16s}")
print('─' * 80)
print(f"{'Baseline (avg 4 models)':45s} {bl_r2:8.4f}          {bl_mae:8.4f}")
for label, summ in [
    ('Multi-Task + OneHot', mt_onehot_summary),
    ('Multi-Task + org/met', mt_cat_reg_summary),
    ('Multi-Task + OpenAI', mt_oai_reg_summary),
]:
    print(f"{label:45s} {summ['r2'][0]:.4f}±{summ['r2'][1]:.4f}  {summ['mae'][0]:.4f}±{summ['mae'][1]:.4f}")

reg_rows = [
    {'Approach': 'Baseline (avg)', 'R2_mean': bl_r2, 'MAE_mean': bl_mae, 'RMSE_mean': bl_rmse, 'Pearson_mean': bl_pr},
    {'Approach': 'Multi-Task OneHot', **{f'{m}_mean': mt_onehot_summary[m][0] for m in ['r2','mae','rmse','pearson']},
     **{f'{m}_std': mt_onehot_summary[m][1] for m in ['r2','mae','rmse','pearson']}},
    {'Approach': 'Multi-Task org/met', **{f'{m}_mean': mt_cat_reg_summary[m][0] for m in ['r2','mae','rmse','pearson']},
     **{f'{m}_std': mt_cat_reg_summary[m][1] for m in ['r2','mae','rmse','pearson']}},
    {'Approach': 'Multi-Task OpenAI', **{f'{m}_mean': mt_oai_reg_summary[m][0] for m in ['r2','mae','rmse','pearson']},
     **{f'{m}_std': mt_oai_reg_summary[m][1] for m in ['r2','mae','rmse','pearson']}},
]
reg_df = pd.DataFrame(reg_rows)

bl_acc = np.mean([v['acc'][0] for v in baseline_cls.values()])
bl_f1w = np.mean([v['f1_weighted'][0] for v in baseline_cls.values()])

cls_rows = [
    {'Approach': 'Baseline (avg)', 'Acc_mean': bl_acc, 'F1w_mean': bl_f1w},
    {'Approach': 'Multi-Task org/met', 'Acc_mean': mt_cat_cls_s['acc'][0], 'Acc_std': mt_cat_cls_s['acc'][1],
     'F1w_mean': mt_cat_cls_s['f1_weighted'][0], 'F1w_std': mt_cat_cls_s['f1_weighted'][1]},
    {'Approach': 'Multi-Task OpenAI', 'Acc_mean': mt_oai_cls_s['acc'][0], 'Acc_std': mt_oai_cls_s['acc'][1],
     'F1w_mean': mt_oai_cls_s['f1_weighted'][0], 'F1w_std': mt_oai_cls_s['f1_weighted'][1]},
]
cls_df = pd.DataFrame(cls_rows)

out_reg = BASE / f'descriptor_multitask_cell6_{DATASET}_regression.xlsx'
out_cls = BASE / f'descriptor_multitask_cell6_{DATASET}_classification.xlsx'
reg_df.to_excel(out_reg, index=False)
cls_df.to_excel(out_cls, index=False)

display(reg_df)
display(cls_df)
print(f'\n✓ {out_reg.name}')
print(f'✓ {out_cls.name}')